### Imports

In [1]:
from pathlib import Path
import json
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

print("PyTorch version :", torch.__version__)
print("CUDA available  :", torch.cuda.is_available())

PyTorch version : 2.14.0+cu126
CUDA available  : True


### Project Paths

In [2]:
current_directory = Path.cwd()
tokenizer_directory = current_directory / "tokenizer"
tokenizer_path = tokenizer_directory / "tokenizer.json"
tokenizer_config_path = tokenizer_directory / "tokenizer_config.json"
print("Current directory :", current_directory)
print("Tokenizer exists  :", tokenizer_path.exists())
print("Config exists     :", tokenizer_config_path.exists())

Current directory : c:\Users\Shivanshu Sirohi\Documents\Multilingual_medical_BERT\stage_1_base_multilingual_model
Tokenizer exists  : True
Config exists     : True


In [3]:
## Load tokenizer
with open(tokenizer_config_path, 'r', encoding='utf-8') as file:
    tokenizer_config = json.load(file)

vocab_size = tokenizer_config['vocab_size']
print("Vocabulary size :", vocab_size)
print("Algorithm       :", tokenizer_config["algorithm"])

Vocabulary size : 21000
Algorithm       : WordPiece


### Model Configuration

In [ ]:
model_config = {
    "vocab_size": vocab_size,          # the vocabulary size that our tokenizer outputs
    "max_sequence_length": 256,        # also called "context window", the lenght of input sequence in one go

    "hidden_size": 128,                # the dimensionality of each token's representation throughout the Transfomer Backbone : [B, 256] to [B, 256, 218]
    "num_hidden_layers": 4,            # number of transformer encoder blocks

    "num_attention_heads": 4,          # multi head attention - 4 different perspectives
    "head_dimension": 32,              # 128 /4 = 32

    "intermediate_size": 256,          # expansion size - The purpose is to give the model a larger intermediate space in which to perfom nonlinear transformations

    "hidden_dropout": 0.05,            # randomly neurons will be switched off in both the FFN layer and Attention layer
    "attention_dropout": 0.05,

    "layer_norm_eps": 1e-5,            # the epsilon in the denominator

    "rope_theta": 10000.0,             # base for rotation of embedding pairs 
    "rotary_dimension": 32,            # all positions of the Q,K matrices will rotate

    "initializer_std": 0.02,           # random weight initializer
}

for key, value in model_config.items():
    print(f"{key:24s}: {value}")

vocab_size              : 21000
max_sequence_length     : 256
hidden_size             : 128
num_hidden_layers       : 4
num_attention_heads     : 4
head_dimension          : 32
intermediate_size       : 256
hidden_dropout          : 0.05
attention_dropout       : 0.05
layer_norm_eps          : 1e-05
num_token_types         : 2
rope_theta              : 10000.0
rotary_dimension        : 32
initializer_std         : 0.02


In [ ]:
assert (
    model_config["hidden_size"]
    == model_config["num_attention_heads"]
    * model_config["head_dimension"]
)

assert model_config["rotary_dimension"] <= model_config["head_dimension"]

assert model_config["rotary_dimension"] % 2 == 0

print("Model configuration is valid.")